In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_excel(r"E:\Data Science by SRK\Machine_learning\project\Taxi_price_prediction_xgboost\cleaned_Taxi_Price_data.xlsx")

In [3]:
df

,Trip_Distance_km,Per_Km_Rate,Trip_Price
0,3.013081,0.800000,36.2624
1,3.634159,1.210000,52.9032
2,3.444576,0.510000,36.4698
3,3.339003,0.630000,15.6180
4,2.265921,1.710000,60.2028
...,...,...,...
946,1.870263,0.620000,34.4049
947,3.849083,0.610000,62.1295
948,2.163323,1.780000,33.1236
949,3.882800,0.820000,61.2090


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 951 entries, 0 to 950
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Trip_Distance_km  951 non-null    float64
 1   Per_Km_Rate       951 non-null    float64
 2   Trip_Price        951 non-null    float64
dtypes: float64(3)
memory usage: 22.4 KB


# X & y

In [5]:
X = df.drop(columns = ['Trip_Price'])
y = df['Trip_Price']

In [6]:
print(X.shape, y.shape)

(951, 2) (951,)


# Train_Test_split

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 5)


# Modelling And Evaluation


Applying hyperparameter tuning for lasso regression

In [8]:
# model
from sklearn.linear_model import ElasticNet
enr_base = ElasticNet()
enr_base.fit(X_train, y_train)

# Predictions
train_predictions = enr_base.predict(X_train)
test_predictions = enr_base.predict(X_test)

# Evaluation
print("Train_R2 : ", enr_base.score(X_train, y_train))
print("Test_R2 : ", enr_base.score(X_test, y_test))

from sklearn.model_selection import cross_val_score
print("Cross_val_score : ", cross_val_score(enr_base, X, y, cv = 5).mean())

Train_R2 :  0.3148901746994418
Test_R2 :  0.3160964987727405
Cross_val_score :  0.3208170248477639


In [9]:
from sklearn.model_selection import GridSearchCV

# model
estimator = ElasticNet()

# parameters & Values
param_grid = {"alpha" : [0.1,0.2,1,2,3,5,10], "l1_ratio" : [0.1,0.5,0.75,0.9,0.95,1]}

# Identifying the best value of the parameter within given values for the given data
model_hp = GridSearchCV(estimator, param_grid, cv= 5, scoring = 'neg_mean_squared_error')
model_hp.fit(X_train, y_train)
model_hp.best_params_

{'alpha': 0.1, 'l1_ratio': 1}

**Rebuilt lasso model using best hyperparameters**

In [10]:
# modelling

enr_best = ElasticNet(alpha=0.1, l1_ratio=1)
enr_best.fit(X_train,y_train)

print("Intercept : ", enr_best.intercept_)
print("coefficient : ", enr_best.coef_)
print("============================================")

# Predictions
train_predictions = enr_best.predict(X_train)
test_predictions  = enr_best.predict(X_test)

# Evaluation
print('Train_R2 : ', enr_best.score(X_train, y_train))
print("Test_R2 : ", enr_best.score(X_test, y_test))
print("cross_val_score : ", cross_val_score(enr_best, X, y, cv=5).mean())

Intercept :  -77.006846893681
coefficient :  [31.55135071 29.25646413]
Train_R2 :  0.4576708790171786
Test_R2 :  0.4321286131181509
cross_val_score :  0.40894812979483997
